In [5]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['TiendaBigData'] 
coleccion = db['AmazonLaptops'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


In [10]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Cierra procesos viejos que hayan quedado abiertos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print(" Limpieza de procesos y temporales completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "Tu_Nombre_de_Grupo"
datos_finales = []   # Se define fuera del try para que siempre exista
driver = None        # Se define fuera del try para poder cerrarlo con seguridad

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"  # Ruta del binario de Chrome

# Argumentos de estabilidad para Docker
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument("--remote-debugging-port=9222")



# User-Agent común
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    # Inicia el navegador
    driver = webdriver.Chrome(options=options)
    print(" Navegador iniciado correctamente.")

    # --- PASO 2: NAVEGACIÓN Y EXTRACCIÓN ---
    limite_paginas = 3
    driver.get("https://www.amazon.es/s?k=laptops")
    #  PAUSA PARA INTERACCIÓN HUMANA
print(" Abre el navegador en http://localhost:6080/vnc.html")
input(" Resuelve CAPTCHA o interactúa con la página y presiona ENTER para continuar...")


    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Página {nivel_pagina + 1} ---")

        # Espera fija para que la página termine de cargar
        time.sleep(10)

        # Espera explícita a que existan resultados
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "div[data-component-type='s-search-result']")
            )
        )

        bloques = driver.find_elements(
            By.CSS_SELECTOR, "div[data-component-type='s-search-result']"
        )

        for bloque in bloques:
            try:
                nombre = bloque.find_element(By.TAG_NAME, "h2").text
                precio = bloque.find_element(By.CSS_SELECTOR, ".a-price-whole").text

                datos_finales.append({
                    "identificador": nombre,
                    "valor": precio,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                })
            except:
                # Si el bloque no tiene nombre o precio, se salta
                continue

        # Intenta avanzar a la siguiente página
        try:
            btn_sig = driver.find_element(By.CLASS_NAME, "s-pagination-next")
            driver.execute_script("arguments[0].click();", btn_sig)
            time.sleep(5)
        except:
            print("No se encontró el botón siguiente o ya es la última página.")
            break

    print(f" Extracción terminada: {len(datos_finales)} productos.")

except Exception as e:
    print(f" Error en Selenium: {e}")

finally:
    # Cierra el navegador solo si logró abrirse
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    client = MongoClient("database", 27017, serverSelectionTimeoutMS=5000)
    db = client["TiendaBigData"]
    coleccion = db["AmazonLaptops"]

    if datos_finales:
        for d in datos_finales:
            # Limpia el valor antes de convertirlo
            v_limpio = str(d["valor"]).replace(".", "").replace(",", "").strip()
            d["valor"] = float(v_limpio) if v_limpio.isdigit() else 0.0

        coleccion.insert_many(datos_finales)
        print(" Datos cargados en MongoDB correctamente.")
    else:
        print(" No hay datos para guardar.")

except Exception as e:
    print(f" Error en MongoDB: {e}")

 Limpieza de procesos y temporales completada.
 Navegador iniciado correctamente.
Sin bloqueo evidente
Abre el entorno visual en http://localhost:6080/vnc.html


Resuelve CAPTCHA o interactúa y presiona ENTER para continuar... 


--- Procesando Página 1 ---
 Error en Selenium: Message: 

 No hay datos para guardar.


In [11]:
# --- PASO 0: LIMPIEZA Y PREPARACIÓN ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Cierra procesos anteriores que hayan quedado abiertos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print(" Limpieza de procesos y temporales completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "Tu_Nombre_de_Grupo"
URL_OBJETIVO = "https://www.amazon.es/s?k=laptops"

datos_finales = []
driver = None

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"

# Argumentos de estabilidad para Docker
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument("--remote-debugging-port=9222")

# User-Agent común
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    # Inicia Chrome
    driver = webdriver.Chrome(options=options)
    print(" Navegador iniciado correctamente.")
    print("🖥 Revisa el navegador en: http://localhost:6080/vnc.html")

    # --- PASO 2: ABRIR LA PÁGINA ---
    driver.get(URL_OBJETIVO)
    time.sleep(8)

    print("Título inicial:", driver.title)
    print("URL inicial:", driver.current_url)

    html = driver.page_source.lower()

    # Diagnóstico básico del estado de la página
    if "captcha" in html or "unusual activity" in html or "temporarily restricted" in html:
        print(" Se detectó una validación o restricción del sitio.")
    else:
        print(" Sin bloqueo evidente en la carga inicial.")

    # Pausa para observación manual
    print(" Abre la interfaz visual en http://localhost:6080/vnc.html")
    input("Cuando veas la página lista para revisar resultados, presiona ENTER para continuar...")

    # Revisión después de la pausa
    print("Título después de la revisión:", driver.title)
    print("URL después de la revisión:", driver.current_url)

    html = driver.page_source.lower()
    if "captcha" in html or "unusual activity" in html or "temporarily restricted" in html:
        print(" El sitio sigue mostrando una validación o restricción.")
        print(" El script no continuará con la extracción.")
    else:
        print(" Se intentará extraer resultados.")

        # --- PASO 3: ESPERAR RESULTADOS ---
        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_all_elements_located(
                    (By.CSS_SELECTOR, "div[data-component-type='s-search-result']")
                )
            )
        except Exception as e:
            print(" No se encontraron resultados con el selector esperado.")
            print("Detalle:", e)
            print("Primeros 1000 caracteres del HTML:")
            print(driver.page_source[:1000])
            raise

        # --- PASO 4: EXTRAER BLOQUES ---
        bloques = driver.find_elements(
            By.CSS_SELECTOR, "div[data-component-type='s-search-result']"
        )

        print(f" Bloques encontrados: {len(bloques)}")

        for bloque in bloques:
            try:
                nombre = bloque.find_element(By.TAG_NAME, "h2").text.strip()
                precio = bloque.find_element(By.CSS_SELECTOR, ".a-price-whole").text.strip()

                if nombre and precio:
                    datos_finales.append({
                        "identificador": nombre,
                        "valor": precio,
                        "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                        "grupo": NOMBRE_GRUPO
                    })
            except:
                continue

        print(f" Extracción terminada: {len(datos_finales)} productos capturados.")

except Exception as e:
    print(f" Error en Selenium: {e}")

finally:
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 5: GUARDAR EN MONGODB ---
try:
    client = MongoClient("database", 27017, serverSelectionTimeoutMS=5000)
    db = client["TiendaBigData"]
    coleccion = db["AmazonLaptops"]

    if datos_finales:
        for d in datos_finales:
            # Limpieza básica del precio
            v_limpio = str(d["valor"]).replace(".", "").replace(",", "").strip()
            d["valor"] = float(v_limpio) if v_limpio.isdigit() else 0.0

        coleccion.insert_many(datos_finales)
        print(" Datos cargados en MongoDB correctamente.")
    else:
        print(" No hay datos para guardar en MongoDB.")

except Exception as e:
    print(f" Error en MongoDB: {e}")

🧹 Limpieza de procesos y temporales completada.
✅ Navegador iniciado correctamente.
🖥️ Revisa el navegador en: http://localhost:6080/vnc.html
Título inicial: Lo sentimos.
URL inicial: https://www.amazon.es/s?k=laptops
✅ Sin bloqueo evidente en la carga inicial.
🧠 Abre la interfaz visual en http://localhost:6080/vnc.html


Cuando veas la página lista para revisar resultados, presiona ENTER para continuar... 


Título después de la revisión: Lo sentimos.
URL después de la revisión: https://www.amazon.es/s?k=laptops
✅ Se intentará extraer resultados.
❌ No se encontraron resultados con el selector esperado.
Detalle: Message: 

Primeros 1000 caracteres del HTML:
<html><head>
    <meta http-equiv="content-type" content="text/html; charset=UTF-8">
    <meta charset="utf-8">
    <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
<title>Lo sentimos.</title>
</head>
<body bgcolor="#FFFFFF" text="#000000">

<!--
Para tratar el acceso automatizado a los datos de Amazon, ponte en contacto con api-services-support@amazon.com.
Para obtener información sobre la migración a nuestras API, consulta nuestras API de Marketplace en https://developer.amazonservices.com/ref=rm_c_sv o nuestra API de publicidad de productos en https://affiliate-program.amazon.com/gp/advertising/api/detail/main.html/ref=rm_c_ac para casos de uso publicitario.

To discuss automated access to Amazon data please contact api-

In [12]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Cierra procesos viejos que hayan quedado abiertos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("🧹 Limpieza de procesos y temporales completada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "Tu_Nombre_de_Grupo"
datos_finales = []   # Se define fuera del try para que siempre exista
driver = None        # Se define fuera del try para poder cerrarlo con seguridad

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()
options.binary_location = "/usr/bin/google-chrome"  # Ruta del binario de Chrome

# Argumentos de estabilidad para Docker
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument("--remote-debugging-port=9222")

# User-Agent común
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    # Inicia el navegador
    driver = webdriver.Chrome(options=options)
    print(" Navegador iniciado correctamente.")

    # --- PASO 2: NAVEGACIÓN Y EXTRACCIÓN ---
    limite_paginas = 3
    driver.get("https://www.amazon.es/s?k=laptops")

    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando Página {nivel_pagina + 1} ---")

        # Espera fija para que la página termine de cargar
        time.sleep(10)

        # Espera explícita a que existan resultados
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located(
                (By.CSS_SELECTOR, "div[data-component-type='s-search-result']")
            )
        )

        bloques = driver.find_elements(
            By.CSS_SELECTOR, "div[data-component-type='s-search-result']"
        )

        for bloque in bloques:
            try:
                nombre = bloque.find_element(By.TAG_NAME, "h2").text
                precio = bloque.find_element(By.CSS_SELECTOR, ".a-price-whole").text

                datos_finales.append({
                    "identificador": nombre,
                    "valor": precio,
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                })
            except:
                # Si el bloque no tiene nombre o precio, se salta
                continue

        # Intenta avanzar a la siguiente página
        try:
            btn_sig = driver.find_element(By.CLASS_NAME, "s-pagination-next")
            driver.execute_script("arguments[0].click();", btn_sig)
            time.sleep(5)
        except:
            print("No se encontró el botón siguiente o ya es la última página.")
            break

    print(f" Extracción terminada: {len(datos_finales)} productos.")

except Exception as e:
    print(f" Error en Selenium: {e}")

finally:
    # Cierra el navegador solo si logró abrirse
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    client = MongoClient("database", 27017, serverSelectionTimeoutMS=5000)
    db = client["TiendaBigData"]
    coleccion = db["AmazonLaptops"]

    if datos_finales:
        for d in datos_finales:
            # Limpia el valor antes de convertirlo
            v_limpio = str(d["valor"]).replace(".", "").replace(",", "").strip()
            d["valor"] = float(v_limpio) if v_limpio.isdigit() else 0.0

        coleccion.insert_many(datos_finales)
        print(" Datos cargados en MongoDB correctamente.")
    else:
        print(" No hay datos para guardar.")

except Exception as e:
    print(f" Error en MongoDB: {e}")
    

🧹 Limpieza de procesos y temporales completada.
✅ Navegador iniciado correctamente.
--- Procesando Página 1 ---
❌ Error en Selenium: Message: 

⚠️ No hay datos para guardar.
